# Conveyor perception

**An end-to-end industrial CV pipeline on a free T4:** real recycling data → trained model → live detection → drift monitoring → triage decisions.

- **Runtime:** Google Colab T4 (free tier, ~12h cap).  
- **Data:** bundled 4-class recycling set (CC BY 4.0).  
- **Model:** YOLO26s, trained in-kernel, cached on re-run.  
- **Goal:** show the loop — train → infer → drift → triage → maintain — on real data, in <5 minutes.


In [ ]:
# --- Cell 1: Runtime + env check ---
import os, sys, json, platform
from pathlib import Path

# --- 1. Colab or local? ---
IN_COLAB = 'google.colab' in sys.modules
print(f'  Runtime: {"Google Colab" if IN_COLAB else "Local (" + platform.node() + ")"}')

# --- 2. Python + key libs (skip import if missing) ---
print(f'  Python: {sys.version.split()[0]}  ({sys.executable.split("/")[-1]})')
for mod in ['numpy', 'torch', 'ultralytics', 'supervision', 'roboflow']:
    try:
        m = __import__(mod)
        v = getattr(m, '__version__', '?')
        print(f'  {mod:14s} {v}')
    except ImportError:
        print(f'  {mod:14s} — not installed yet (cell 2 will install)')

# --- 3. GPU (or warn if CPU-only) ---
_gpu = 'unknown'
try:
    import torch
    _gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'
except Exception:
    pass
print(f'  GPU:    {_gpu}')

# --- 4. Disk + RAM ---
_disk_free = '?'
try:
    import shutil
    _u = shutil.disk_usage('/')
    _disk_free = f'{_u.free / 1e9:.1f} GB free of {_u.total / 1e9:.1f} GB'
except Exception:
    pass
print(f'  Disk:   {_disk_free}')

# --- 5. Locate the repo (for local runs) — Colab gets cloned by cell 2 ---
REPO = Path('/content/conveyor-perception' if IN_COLAB else '.').resolve()
if not IN_COLAB:
    # Walk up until we find the repo root (contains pyproject.toml)
    while not (REPO / 'pyproject.toml').exists() and REPO != REPO.parent:
        REPO = REPO.parent
print(f'  Repo:   {REPO}{" (will be cloned here by cell 2)" if IN_COLAB else ""}')

# NOTE: colab_session + the state singleton are intentionally NOT used here.
# Cell 1 runs BEFORE the clone (cell 2), so the repo isn't on disk yet — any
# `import colab_session` would crash with ModuleNotFoundError. State init is
# deferred to cell 3, which runs after the clone + install are done. (Aug 22 2026)
print()
print('  ✓ env check done.  Next: cell 2 (clone + install).')
